# Step 6 (Baseline) — RUL BiLSTM, no augmentation, no multi-task

**RESS 2025 — GAN-Conformal-RUL**

Establishes the **anchor RMSE**: a 2-layer BiLSTM with temporal attention and a single RUL
regression head, trained on real data only. Every later contribution — GAN augmentation, the
stage-classification head, RUL-loss masking — is measured against this number.

| | |
|---|---|
| **Encoder** | 2-layer BiLSTM (hidden 64/direction) + temporal attention |
| **Head** | RUL regression → sigmoid → [0,1] |
| **Loss** | MSE on normalised RUL |
| **Trained on** | real training windows only |
| **Reported** | RMSE & MAE in **minutes** (primary), normalised RMSE (secondary) |

Minutes are recovered by multiplying each window's normalised RUL by its bearing's lifetime —
the unit used across the XJTU-SY literature (Lu et al. 2022 cumulative RMSE 21.90 min is the
target to beat).

## 1. Setup

In [ ]:
import os, sys, shutil
os.chdir('/content')
REPO_PATH = '/content/RESS_2025_GAN_Conformal_RUL'
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)
!git clone https://github.com/f-khadija-benzine/RESS_2025_GAN_Conformal_RUL.git {REPO_PATH}
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH); sys.path.insert(0, f'{REPO_PATH}/src')
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = f'{REPO_PATH}/figures'; os.makedirs(SAVE_DIR, exist_ok=True)
import torch
print('CUDA:', torch.cuda.is_available())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import XJTUSYLoader
from health_indicator_v3 import HealthIndicatorPipeline
from windowing import prepare_all_folds, build_folds, WINDOW_SIZE
from model import (ModelConfig, RULTrainer, rul_metrics,
                   per_bearing_rmse, cumulative_rmse, phm_score)

## 2. Rebuild data (Steps 2–3, log_clip scaling)

In [ ]:
CANDIDATES = ['/content/drive/MyDrive/XJTU-SY',
              '/content/drive/MyDrive/XJTU-SY_Bearing_Datasets',
              '/content/drive/MyDrive/data/XJTU-SY']
DATA_ROOT = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert DATA_ROOT, f'not found: {CANDIDATES}'

all_data = XJTUSYLoader(DATA_ROOT).load_all()
pipeline = HealthIndicatorPipeline(fpt_consecutive=5, fpt_min_relative_rise=0.20)
results = pipeline.process_all(all_data, verbose=False)

fold_data = prepare_all_folds(results, scaling_method='log_clip', verbose=False)
folds = build_folds(results)
print(f'{len(fold_data)} folds ready.')

## 3. Bearing lifetimes for the minutes conversion

Each test window's normalised RUL is converted to minutes by multiplying by its bearing's
total lifetime. We build a per-window lifetime vector for each fold's test set, in the same
window order that `prepare_fold` produced (bearings concatenated in the order listed in the
fold's `test` field).

In [ ]:
# Per-window metadata for the test set of a fold, matching prepare_fold's
# concatenation order (bearings in the order listed in fold['test']).
def test_meta(fold, results, window_size=WINDOW_SIZE):
    """Returns per-window (lifetime_min, bearing_id) arrays and a dict of
    prognostic durations dT = t_EOL - t_FPT + 1 per bearing."""
    life, bids = [], []
    dT = {}
    for bid in fold['test']:
        r = results[bid]
        n_rec = r['features'].shape[0]
        n_win = max(0, n_rec - window_size + 1)
        life.extend([n_rec] * n_win)
        bids.extend([bid] * n_win)
        # prognostic duration: FPT to EOL. results carries fpt index per bearing.
        fpt = r.get('fpt', 0)
        dT[bid] = n_rec - fpt + 1
    return np.array(life, float), np.array(bids), dT

# sanity: metadata length must equal test window count
for d, f in zip(fold_data, folds):
    life, bids, _ = test_meta(f, results)
    assert len(life) == len(d['X_test']) == len(bids), \
        f"fold {f['fold']}: {len(life)} meta vs {len(d['X_test'])} windows"
print('per-window metadata aligned with test windows for all folds.')

## 4. Train the baseline, one fold at a time

Train on real training windows, early-stop on validation RMSE, evaluate on the held-out test
bearings. RUL target only — the stage labels are ignored for the baseline.

In [ ]:
cfg = ModelConfig(epochs=100, patience=15, lr=1e-3)

rows = []
per_bearing_all = {}     # {fold: {bearing: rmse}}
trainers = {}
for d, f in zip(fold_data, folds):
    k = d['fold']
    print(f"\n{'='*56}\nFOLD {k}\n{'='*56}")
    tr = RULTrainer(cfg)
    tr.fit(d['X_train'], d['y_rul_train'],
           d['X_val'],   d['y_rul_val'], verbose=True)
    trainers[k] = tr

    y_pred = tr.predict(d['X_test'])
    life, bids, dT = test_meta(f, results)
    stages = d['y_stage_test']   # 0-indexed; +1 not needed, we compare >=2 below
    # NOTE: y_stage is 0-indexed (0,1,2). Past-FPT = stage index >= 1.
    stages_1idx = stages + 1     # -> 1,2,3 to match per_bearing_rmse convention

    # (a) pooled minutes + normalised
    m = rul_metrics(d['y_rul_test'], y_pred, life)
    # (b) per-bearing RMSE (post-FPT only, as Lu et al. score the FPT-EOL span)
    pb = per_bearing_rmse(d['y_rul_test'], y_pred, bids, life,
                          stages=stages_1idx, post_fpt_only=True)
    per_bearing_all[k] = {b: v for b, v in pb.items() if b != '_mean'}
    # (c) cumulative dT-weighted RMSE (directly comparable to Lu et al.)
    cum = cumulative_rmse(pb, dT)
    # (d) PHM 2012 score (higher better, in (0,1]; penalises late predictions)
    sc = phm_score(d['y_rul_test'], y_pred, bids)

    m.update({'fold': k, 'per_bearing_mean': pb.get('_mean', float('nan')),
              'cumulative': cum, 'phm_score': sc.get('_score', float('nan'))})
    rows.append(m)
    print(f"  TEST  pooled-RMSE {m['rmse_min']:.2f} min | "
"
"          f"per-bearing {pb.get('_mean', float('nan')):.2f} | "
"
"          f"cumulative {cum:.2f} | norm {m['rmse_norm']:.4f} | "
          f"score {sc.get('_score', float('nan')):.3f}")

## 5. Baseline results — the anchor

In [ ]:
df = pd.DataFrame(rows)[['fold','cumulative','rmse_norm','mae_norm','r2','phm_score']]
df.columns = ['fold','cumul_min','norm_rmse','norm_mae','r2','phm_score']
print(df.to_string(index=False))
print('-'*60)
print(f"  MEAN cumulative (Lu-comparable): {df['cumulative_min'].mean():6.2f} min")
print(f"  MEAN normalised RMSE:            {df['norm_rmse'].mean():.4f}  (BiLSTM benchmark ~0.158)")
print(f"  MEAN normalised MAE:             {df['norm_mae'].mean():.4f}  (BiLSTM benchmark ~0.126)")
print(f"  MEAN R2:                         {df['r2'].mean():.4f}")
print(f"  MEAN PHM score (higher better):  {df['phm_score'].mean():.3f}")
print(f"\nLu et al. 2022 cumulative RMSE: 21.90 (HP-JT, their best) | 31.00 (HP, plain predictor)")
print(f"Compare our baseline cumulative to their PLAIN predictor (31.00) — the fair baseline match.")

In [ ]:
# Training curves — confirm clean convergence and that early stopping fired sensibly
fig, axes = plt.subplots(1, len(trainers), figsize=(4*len(trainers), 3), sharey=True)
if len(trainers) == 1: axes = [axes]
for ax, (k, tr) in zip(axes, trainers.items()):
    ax.plot(tr.history['train_rmse'], label='train', lw=1.2)
    ax.plot(tr.history['val_rmse'], label='val', lw=1.2)
    ax.set_title(f'Fold {k}'); ax.set_xlabel('epoch'); ax.grid(alpha=0.3)
    if k == list(trainers)[0]: ax.set_ylabel('normalised RMSE'); ax.legend(fontsize=8)
plt.suptitle('Baseline training curves', y=1.03)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/06_baseline_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Next

This baseline is the anchor. The next iterations layer on, one at a time, each as an ablation row:

1. **+GAN** — load frozen generators from Drive, augment the training set with synthetic
   near-failure windows at `augment_ratio`, retrain, compare RMSE. *Tests C1.*
2. **+Multi-task** — add the stage-classification head and the combined loss. *Tests C2.*
3. **+RUL masking** — mask the RUL loss to post-FPT windows. *The deferred Step 3 decision.*

Keeping them incremental means any change in RMSE is attributable to exactly one thing.